# MyFirstNEURON — Colab Edition (Prototype 2)

This notebook is a Google-Colab-friendly, Python/Jupyter port of **MyFirstNEURON**, a NEURON demo
by Arthur Houweling and Terry Sejnowski (Salk Institute), based on experiments from
*Electrophysiology of the Neuron* by Huguenard & McCormick. Original files:
https://modeldb.science/3808

**Status:** experiments 1-6 from the original "Basics" menu (resting potential, membrane
properties, impulse generation), experiments 7-9 from the "Fast Na,K" voltage-clamp menu,
experiments 11-15 from the "Other" menu (iA, iL, iC, iAHP, iT, iM currents), and experiments
16-17 from the "Synaptic" menu are all implemented, sharing a single cell plus common
simulation/dashboard helpers. In the original GUI each menu entry covers one or more textbook
experiments, which is why there are fewer presets than textbook experiment numbers.

No local installation is needed — just run the cells top to bottom in Colab.

# 1. Setup

## 1.1 Installation (run once per Colab session)

Installs the `neuron` Python package, clones the original `.mod` mechanism files from the
[ModelDB GitHub mirror](https://github.com/ModelDBRepository/3808), and compiles them with
`nrnivmodl`.

In [ ]:
%%capture
!pip install neuron

In [ ]:
import os

MOD_SRC_DIR = "mfn_src"

if not os.path.isdir(MOD_SRC_DIR):
    !git clone --depth 1 https://github.com/ModelDBRepository/3808.git {MOD_SRC_DIR}

!cd {MOD_SRC_DIR} && nrnivmodl

In [ ]:
from neuron import n
from neuron import load_mechanisms
import matplotlib.pyplot as plt

load_mechanisms(MOD_SRC_DIR)
n.load_file("stdrun.hoc")

print("NEURON is ready, mechanisms loaded from:", MOD_SRC_DIR)

## 1.2 Build the cell

A single spherical compartment (same geometry as the original demo: total membrane area of
29000 &mu;m&sup2;), with passive leak channels (Na/K/Ca/Cl/Mg) and Hodgkin-Huxley
sodium/potassium channels. All experiments 1-6 reuse this same cell; only the parameter
values differ.

In [ ]:
import math

soma = n.Section(name="soma")
soma.L = 290 / math.pi
soma.diam = 100
soma.nseg = 1

for mech in ("leak", "HH", "iA", "iL", "iT", "iC", "iAHP", "iM", "cadyn"):
    soma.insert(mech)

# calcium-shell decay constants for cadyn, fixed across every experiment (never overridden
# by any e*.par file): a 1 ms relaxation towards 5e-5 mM, pump disabled (kt=0)
soma(0.5).taur_cadyn = 1.0
soma(0.5).cainf_cadyn = 5e-5
soma(0.5).kt_cadyn = 0.0
soma(0.5).kd_cadyn = 5e-5

# celsius is set per-experiment (see _apply_cell_params below); e7.par uses 23.5 instead of 35
print("soma area (um^2):", n.area(0.5, sec=soma))

## 1.3 Common recording

`t_vec`/`v_vec` are shared by every experiment type; the current-clamp `IClamp` (section 1) and
voltage-clamp `VClamp` (section 2) each add their own stimulus object and extra recordings.


In [ ]:
t_vec = n.Vector().record(n._ref_t)
v_vec = n.Vector().record(soma(0.5)._ref_v)

## 1.4 Shared simulation helpers

`DEFAULT_PARAMS` holds the cell/ion-concentration baseline shared by every experiment type.
Each experiment section below layers its own stimulus-specific defaults on top (e.g. the
current-clamp `stim_delay`/`stim_dur`/`stim_amp`, or the voltage-clamp `vc_dur*`/`vc_amp*`),
so callers only pass whichever values differ from baseline. `_apply_cell_params()` and
`_plot_panels()` are reused by both `run_experiment()` (section 1) and `run_vclamp_experiment()`
(section 2) to avoid duplicating that logic.


In [ ]:
# baseline cell/ion values shared by every experiment type
DEFAULT_PARAMS = dict(
    pna_leak=0.0, pk_leak=0.0,
    gnabar_HH=0.069, gkbar_HH=0.0069,
    gkbar_iA=0.0,
    pcabar_iL=0.0, pcabar_iT=0.0, gkbar_iC=0.0, gkbar_iAHP=0.0, gkbar_iM=0.0,
    cai0_ca_ion=5e-5, cao0_ca_ion=2.0,
    nai=31.0, nao=145.0, ki=135.0, ko=3.1,
    cli=7.0, clo=120.0, mgo=1.0,
    celsius=35.0,
    v_init=-65.0, tstop=20.0,
)

def _apply_cell_params(p):
    soma(0.5).pna_leak = p["pna_leak"]
    soma(0.5).pk_leak = p["pk_leak"]
    soma(0.5).gnabar_HH = p["gnabar_HH"]
    soma(0.5).gkbar_HH = p["gkbar_HH"]
    soma(0.5).gkbar_iA = p["gkbar_iA"]
    soma(0.5).pcabar_iL = p["pcabar_iL"]
    soma(0.5).pcabar_iT = p["pcabar_iT"]
    soma(0.5).gkbar_iC = p["gkbar_iC"]
    soma(0.5).gkbar_iAHP = p["gkbar_iAHP"]
    soma(0.5).gkbar_iM = p["gkbar_iM"]
    n.cai0_ca_ion = p["cai0_ca_ion"]
    n.cao0_ca_ion = p["cao0_ca_ion"]
    soma(0.5).nai = p["nai"]
    soma(0.5).nao = p["nao"]
    soma(0.5).ki = p["ki"]
    soma(0.5).ko = p["ko"]
    soma(0.5).cli = p["cli"]
    soma(0.5).clo = p["clo"]
    soma(0.5).mgo = p["mgo"]
    n.celsius = p["celsius"]

def _reset_stimuli():
    """Neutralize every stimulus/synapse object shared across experiment sections so that
    switching sections never leaves a stale IClamp/VClamp/synapse active (mirrors
    reset_soma_pars() in the original hoc, which zeroes all of these before every experiment)."""
    if "stim" in globals():
        stim.delay = stim.dur = stim.amp = 0
    if "stim2" in globals():
        stim2.delay = stim2.dur = stim2.amp = 0
    if "vclamp" in globals():
        vclamp.dur[0] = vclamp.dur[1] = vclamp.dur[2] = 0
        vclamp.amp[0] = vclamp.amp[1] = vclamp.amp[2] = 0
    if "ampasyn" in globals():
        ampasyn.gmaxEPSP = nmdasyn.gmaxEPSP = 0
        ampasyn.w = nmdasyn.w = 0
        gabaAsyn.gmaxIPSP = gabaBsyn.gmaxIPSP = 0
        gabaAsyn.w = gabaBsyn.w = 0

def _panel_data(spec):
    # "compute" panels are derived quantities (e.g. ina+ik) recomputed after every run
    return spec["compute"]() if "compute" in spec else spec["vector"]

def _plot_panels(plot_panels, keys=None):
    keys = list(plot_panels.keys()) if keys is None else list(keys)
    if not keys:
        print("No panels selected.")
        return

    fig, axes = plt.subplots(len(keys), 1, figsize=(7, 2.5 * len(keys)), sharex=True)
    axes = [axes] if len(keys) == 1 else axes
    for ax, key in zip(axes, keys):
        spec = plot_panels[key]
        ax.plot(t_vec, _panel_data(spec))
        ax.set_ylabel(spec["ylabel"])
        if spec.get("ylim"):
            ax.set_ylim(*spec["ylim"])
    axes[-1].set_xlabel("time (ms)")
    plt.show()


## 1.5 Shared dashboard helpers

Both the current-clamp and voltage-clamp dashboards below need the same slider/reset/changed-checkbox
row per parameter, and the same panel-visibility checkboxes. `_build_param_rows()` and
`_build_panel_checks()` factor that out so each dashboard only wires up its own presets and run function.


In [ ]:
from ipywidgets import FloatSlider, Checkbox, Button, Dropdown, HBox, VBox, Output, Layout, Label
from IPython.display import display

def _display_value(param_ui, name, raw_value):
    return raw_value / param_ui[name].get("scale", 1)

def _raw_value(param_ui, name, display_value):
    return display_value * param_ui[name].get("scale", 1)

def _raw_values(param_ui, sliders):
    return {name: _raw_value(param_ui, name, slider.value) for name, slider in sliders.items()}

def _build_param_rows(param_ui, get_default):
    """get_default(name) -> current default/raw value for that parameter (may change over time)."""
    sliders, changed_flags, rows = {}, {}, []

    def _make_change_handler(name):
        def _on_change(change):
            changed_flags[name].value = (change["new"] != _display_value(param_ui, name, get_default(name)))
        return _on_change

    def _make_reset_handler(name):
        def _on_click(_btn):
            sliders[name].value = _display_value(param_ui, name, get_default(name))
        return _on_click

    for name, spec in param_ui.items():
        slider_kwargs = {k: v for k, v in spec.items() if k not in ("scale", "unit", "description")}
        slider = FloatSlider(value=_display_value(param_ui, name, get_default(name)), description=spec["description"],
                              layout=Layout(width="350px"), **slider_kwargs)
        unit_label = Label(value=spec.get("unit", ""), layout=Layout(width="110px"))
        changed = Checkbox(value=False, description="changed", disabled=True, indent=False,
                            layout=Layout(width="90px"))
        reset_btn = Button(description="reset", layout=Layout(width="60px"))

        sliders[name] = slider
        changed_flags[name] = changed
        slider.observe(_make_change_handler(name), names="value")
        reset_btn.on_click(_make_reset_handler(name))

        rows.append(HBox([slider, unit_label, changed, reset_btn]))

    return sliders, changed_flags, rows

def _build_panel_checks(plot_panels):
    checks = {key: Checkbox(value=True, description=spec["label"], indent=False, layout=Layout(width="90px"))
              for key, spec in plot_panels.items()}
    return checks, HBox([Label("Show:")] + list(checks.values()))

def _selected_panels(panel_checks):
    return [key for key, cb in panel_checks.items() if cb.value]


# 2. Basic Experiments 1-6

Each entry only lists the values that differ from `DEFAULT_PARAMS`, mirroring how the original
`e*.par` files only set the parameters relevant to that experiment. In the original GUI's
"Basics" menu, each entry covers two textbook experiments (hence "1,2" etc. in the labels).
Sourced directly from `e1.par`, `e3.par` and `e5.par`.

In [ ]:
# current-clamp stimulus, specific to experiments 1-6
stim = n.IClamp(soma(0.5))
i_vec = n.Vector().record(stim._ref_i)

# current-clamp-specific defaults, layered on top of the common cell/ion defaults
DEFAULT_CCLAMP_PARAMS = {**DEFAULT_PARAMS, "stim_delay": 0.0, "stim_dur": 0.0, "stim_amp": 0.0}

PLOT_PANELS = {
    "stim": dict(label="current", vector=i_vec, ylabel="injected current (nA)", ylim=(-1, 5)),
    "v": dict(label="voltage", vector=v_vec, ylabel="membrane potential (mV)", ylim=(-100, 50)),
}

def run_experiment(panels=None, **overrides):
    p = {**DEFAULT_CCLAMP_PARAMS, **overrides}
    _apply_cell_params(p)
    _reset_stimuli()  # a previous vclamp/synaptic run must not stay active here

    stim.delay = p["stim_delay"]
    stim.dur = p["stim_dur"]
    stim.amp = p["stim_amp"]

    n.tstop = p["tstop"]
    n.v_init = p["v_init"]
    n.run()


    _plot_panels(PLOT_PANELS, panels)

In [ ]:
# @title Specifying default parameters

EXPERIMENTS = {
    "1: Resting potential (exp. 1,2)": dict(
        pna_leak=2.07e-7, pk_leak=3.45e-6,
        gnabar_HH=0.0, gkbar_HH=0.0,
        nai=31, nao=145, ki=135, ko=3.1,
        v_init=-65, tstop=20,
    ),
    "2: Membrane properties (exp. 3,4)": dict(
        pna_leak=2.07e-7, pk_leak=3.45e-6,
        nai=31, nao=145, ki=135, ko=3.1,
        stim_delay=10, stim_dur=50, stim_amp=1.5,
        v_init=-65, tstop=80,
    ),
    "3: Impulse generation (exp. 5,6)": dict(
        pna_leak=2.07e-7, pk_leak=3.45e-6,
        nai=31, nao=145, ki=135, ko=3.1,
        stim_delay=10, stim_dur=50, stim_amp=2,
        v_init=-65, tstop=80,
    ),
}

# UI metadata (ranges/labels), wide enough to cover every experiment preset above.
# "scale" (default 1) lets a slider show nicer numbers than the raw NEURON value:
# browser range sliders handle small values like 1e-7 poorly, so pna_leak/pk_leak
# are shown as plain numbers and converted back to real units before running.
PARAM_UI = {
    "pna_leak": dict(min=0, max=8, step=0.01, readout_format=".2f", description="pNa (leak)", unit="\u00d710\u207b\u2077 cm/s", scale=1e-7),
    "pk_leak": dict(min=0, max=8, step=0.01, readout_format=".2f", description="pK (leak)", unit="\u00d710\u207b\u2076 cm/s", scale=1e-6),
    "gnabar_HH": dict(min=0, max=0.15, step=0.001, readout_format=".3f", description="gNa (HH)", unit="S/cm\u00b2"),
    "gkbar_HH": dict(min=0, max=0.02, step=0.0005, readout_format=".4f", description="gK (HH)", unit="S/cm\u00b2"),
    "nai": dict(min=0, max=60, step=1, description="nai", unit="mM"),
    "nao": dict(min=0, max=200, step=1, description="nao", unit="mM"),
    "ki": dict(min=0, max=200, step=1, description="ki", unit="mM"),
    "ko": dict(min=0, max=10, step=0.1, description="ko", unit="mM"),
    "stim_delay": dict(min=0, max=50, step=1, description="stim delay", unit="ms"),
    "stim_dur": dict(min=0, max=100, step=1, description="stim dur", unit="ms"),
    "stim_amp": dict(min=0, max=5, step=0.1, description="stim amp", unit="nA"),
    "v_init": dict(min=-90, max=-30, step=1, description="v_init", unit="mV"),
    "tstop": dict(min=5, max=200, step=5, description="tstop", unit="ms"),
}

def build_basicExp_dashboard():
    def _preset_for(exp_name):
        return {**DEFAULT_CCLAMP_PARAMS, **EXPERIMENTS[exp_name]}

    exp_selector = Dropdown(options=list(EXPERIMENTS.keys()), description="Experiment:",
                             style={"description_width": "initial"}, layout=Layout(width="320px"))
    current_defaults = _preset_for(exp_selector.value)

    panel_checks, panels_row = _build_panel_checks(PLOT_PANELS)
    sliders, changed_flags, rows = _build_param_rows(PARAM_UI, lambda name: current_defaults[name])

    run_button = Button(description="Run", button_style="success", icon="play")
    reset_all_button = Button(description="Reset all", icon="undo")
    output = Output()

    def _on_run(_btn):
        with output:
            output.clear_output(wait=True)
            run_experiment(panels=_selected_panels(panel_checks), **_raw_values(PARAM_UI, sliders))

    def _on_reset_all(_btn):
        for name, slider in sliders.items():
            slider.value = _display_value(PARAM_UI, name, current_defaults[name])

    def _on_experiment_change(change):
        nonlocal current_defaults
        current_defaults = _preset_for(change["new"])
        _on_reset_all(None)
        _on_run(None)

    run_button.on_click(_on_run)
    reset_all_button.on_click(_on_reset_all)
    exp_selector.observe(_on_experiment_change, names="value")

    dashboard = VBox([exp_selector, HBox([VBox([panels_row, output]),
                                          VBox([HBox([run_button, reset_all_button])] + rows)])])
    _on_run(None)  # show a default run immediately, without waiting for a click
    return dashboard


Pick an experiment from the dropdown to load its preset, then adjust sliders and click **Run**. Each parameter has a **reset** button to restore the current
experiment's value, and a **changed** checkbox that ticks itself whenever the value differs
from that preset.

In [ ]:
from IPython.display import display
display(build_basicExp_dashboard())

# 3. Voltage clamp (exp. 7-9)

Instead of injecting current and watching voltage respond, a voltage clamp commands the
membrane potential directly (NEURON's `VClamp`) and records the resulting transmembrane
current. `VClamp` steps through three successive commanded potentials `amp0`/`amp1`/`amp2`,
held for durations `dur0`/`dur1`/`dur2`: e7.par first holds at -100 mV (settling potential),
then steps to 0 mV to evoke the fast Na and delayed-rectifier K currents. Sourced directly
from `e7.par` (note it also uses a lower simulation temperature, 23.5 &deg;C, than the
current-clamp experiments).

Textbook experiments 8 and 9 reuse this same protocol but isolate one current at a time by
zeroing the other conductance: set **gK (HH)** to 0 to isolate the transient Na current, or
**gNa (HH)** to 0 to isolate the delayed-rectifier K current.


In [ ]:
# voltage-clamp stimulus, specific to experiments 7-9
vclamp = n.VClamp(soma(0.5))

ina_vec = n.Vector().record(soma(0.5)._ref_ina)
ik_vec = n.Vector().record(soma(0.5)._ref_ik)

# voltage-clamp-specific defaults, layered on top of the common cell/ion defaults
DEFAULT_VCLAMP_PARAMS = {**DEFAULT_PARAMS,
    "vc_dur0": 0.0, "vc_dur1": 0.0, "vc_dur2": 0.0,
    "vc_amp0": 0.0, "vc_amp1": 0.0, "vc_amp2": 0.0,
}

# "v" reuses the shared v_vec (it will show the commanded steps); "im" is a derived
# quantity (ina+ik), recomputed after every run via "compute" instead of a static vector
PLOT_PANELS_VC = {
    "v": dict(label="voltage", vector=v_vec, ylabel="command potential (mV)", ylim=(-100, 50)),
    "im": dict(label="ina+ik", compute=lambda: ina_vec.c().add(ik_vec),
               ylabel="ina + ik (mA/cm\u00b2)", ylim=(-1, 1)),
}

def run_vclamp_experiment(panels=None, **overrides):
    p = {**DEFAULT_VCLAMP_PARAMS, **overrides}
    _apply_cell_params(p)
    _reset_stimuli()  # a previous current-clamp/synaptic run must not stay active here

    vclamp.dur[0] = p["vc_dur0"]
    vclamp.dur[1] = p["vc_dur1"]
    vclamp.dur[2] = p["vc_dur2"]
    vclamp.amp[0] = p["vc_amp0"]
    vclamp.amp[1] = p["vc_amp1"]
    vclamp.amp[2] = p["vc_amp2"]

    n.tstop = p["tstop"]
    n.v_init = p["v_init"]
    n.run()

    _plot_panels(PLOT_PANELS_VC, panels)

In [ ]:
# @title Specifying default parameters

VCLAMP_EXPERIMENTS = {
    "4: Voltage clamp (exp. 7,8,9)": dict(
        gnabar_HH=0.0345, gkbar_HH=0.0069,
        nai=30, nao=145, ki=135, ko=3.1,
        celsius=23.5,
        vc_dur0=10, vc_dur1=10, vc_amp0=-100, vc_amp1=0, vc_amp2=-100,
        v_init=-65, tstop=20,
    ),
}

VC_PARAM_UI = {
    "celsius": dict(min=0, max=40, step=0.5, description="celsius", unit="\u00b0C"),
    "pna_leak": dict(min=0, max=8, step=0.01, readout_format=".2f", description="pNa (leak)", unit="\u00d710\u207b\u2077 cm/s", scale=1e-7),
    "pk_leak": dict(min=0, max=8, step=0.01, readout_format=".2f", description="pK (leak)", unit="\u00d710\u207b\u2076 cm/s", scale=1e-6),
    "gnabar_HH": dict(min=0, max=0.15, step=0.001, readout_format=".3f", description="gNa (HH)", unit="S/cm\u00b2"),
    "gkbar_HH": dict(min=0, max=0.02, step=0.0005, readout_format=".4f", description="gK (HH)", unit="S/cm\u00b2"),
    "nai": dict(min=0, max=60, step=1, description="nai", unit="mM"),
    "nao": dict(min=0, max=200, step=1, description="nao", unit="mM"),
    "ki": dict(min=0, max=200, step=1, description="ki", unit="mM"),
    "ko": dict(min=0, max=10, step=0.1, description="ko", unit="mM"),
    "vc_dur0": dict(min=0, max=50, step=1, description="dur0 (hold)", unit="ms"),
    "vc_dur1": dict(min=0, max=50, step=1, description="dur1 (step)", unit="ms"),
    "vc_dur2": dict(min=0, max=50, step=1, description="dur2 (return)", unit="ms"),
    "vc_amp0": dict(min=-100, max=50, step=1, description="amp0 (hold)", unit="mV"),
    "vc_amp1": dict(min=-100, max=50, step=1, description="amp1 (step)", unit="mV"),
    "vc_amp2": dict(min=-100, max=50, step=1, description="amp2 (return)", unit="mV"),
    "v_init": dict(min=-90, max=-30, step=1, description="v_init", unit="mV"),
    "tstop": dict(min=5, max=200, step=5, description="tstop", unit="ms"),
}

def build_vclampExp_dashboard():
    # only one preset for now, so no experiment selector (unlike the current-clamp dashboard)
    defaults = {**DEFAULT_VCLAMP_PARAMS, **VCLAMP_EXPERIMENTS["4: Voltage clamp (exp. 7,8,9)"]}

    panel_checks, panels_row = _build_panel_checks(PLOT_PANELS_VC)
    sliders, changed_flags, rows = _build_param_rows(VC_PARAM_UI, lambda name: defaults[name])

    run_button = Button(description="Run", button_style="success", icon="play")
    reset_all_button = Button(description="Reset all", icon="undo")
    output = Output()

    def _on_run(_btn):
        with output:
            output.clear_output(wait=True)
            run_vclamp_experiment(panels=_selected_panels(panel_checks), **_raw_values(VC_PARAM_UI, sliders))

    def _on_reset_all(_btn):
        for name, slider in sliders.items():
            slider.value = _display_value(VC_PARAM_UI, name, defaults[name])

    run_button.on_click(_on_run)
    reset_all_button.on_click(_on_reset_all)

    dashboard = VBox([HBox([VBox([panels_row, output]),
                             VBox([HBox([run_button, reset_all_button])] + rows)])])
    _on_run(None)  # show a default run immediately, without waiting for a click
    return dashboard


Adjust the holding/step potentials and durations and click **Run**. Try zeroing **gK (HH)**
or **gNa (HH)** to isolate the individual current underlying the combined `ina+ik` trace.


In [ ]:
display(build_vclampExp_dashboard())

# 4. Other currents (exp. 11-15)

Six additional currents are inserted onto the same `soma` alongside `leak`/`HH`/`iA`: the
high-threshold Ca<sup>2+</sup> current `iL`, the low-threshold Ca<sup>2+</sup> current `iT`
(a "dummy ion" current that does not itself change `cai`), the Ca<sup>2+</sup>-activated K<sup>+</sup>
currents `iC` and `iAHP`, the voltage-dependent K<sup>+</sup> current `iM`, and `cadyn`
(submembrane Ca<sup>2+</sup> concentration decay, needed for `iL`/`iC`/`iAHP` to interact via `cai`).
Each preset below sets the relevant conductance(s) to a nonzero value and leaves the others at 0,
isolating one current at a time (exactly like the original "Other" experiments menu). Two current-clamp
electrodes are available here: `stim` (a delayed pulse) and `stim2` (a constant "base" current, used
by the `iT` burst-firing preset to hold the cell hyperpolarized). Sourced directly from `e11a.par`,
`e11b.par`, `e12.par`, `e13.par`, `e14.par`, `e15a.par` and `e15b.par`.



In [ ]:
# second current-clamp electrode, specific to experiments 11-15 (e14.par's constant "base"
# current alongside the pulse on `stim`); `_reset_stimuli()` already knows to neutralize it
stim2 = n.IClamp(soma(0.5))

ik_iA_vec = n.Vector().record(soma(0.5)._ref_ik_iA)
ica_iL_vec = n.Vector().record(soma(0.5)._ref_ica_iL)
iCa_iT_vec = n.Vector().record(soma(0.5)._ref_iCa_iT)
ik_iC_vec = n.Vector().record(soma(0.5)._ref_ik_iC)
ik_iAHP_vec = n.Vector().record(soma(0.5)._ref_ik_iAHP)
ik_iM_vec = n.Vector().record(soma(0.5)._ref_ik_iM)
cai_vec = n.Vector().record(soma(0.5)._ref_cai)

# "other-currents"-specific defaults, layered on top of the common cell/ion defaults
DEFAULT_OTHER_PARAMS = {**DEFAULT_PARAMS,
    "stim_delay": 0.0, "stim_dur": 0.0, "stim_amp": 0.0,
    "stim2_delay": 0.0, "stim2_dur": 0.0, "stim2_amp": 0.0,
    "vc_dur0": 0.0, "vc_dur1": 0.0, "vc_dur2": 0.0,
    "vc_amp0": 0.0, "vc_amp1": 0.0, "vc_amp2": 0.0,
}

PLOT_PANELS_OTHER = {
    "v": dict(label="voltage", vector=v_vec, ylabel="membrane potential (mV)", ylim=(-100, 50)),
    "iA": dict(label="iA", vector=ik_iA_vec, ylabel="ik_iA (mA/cm\u00b2)"),
    "iL": dict(label="iL", vector=ica_iL_vec, ylabel="ica_iL (mA/cm\u00b2)"),
    "iT": dict(label="iT", vector=iCa_iT_vec, ylabel="iCa_iT (mA/cm\u00b2)"),
    "iC": dict(label="iC", vector=ik_iC_vec, ylabel="ik_iC (mA/cm\u00b2)"),
    "iAHP": dict(label="iAHP", vector=ik_iAHP_vec, ylabel="ik_iAHP (mA/cm\u00b2)"),
    "iM": dict(label="iM", vector=ik_iM_vec, ylabel="ik_iM (mA/cm\u00b2)"),
    "cai": dict(label="cai", vector=cai_vec, ylabel="[Ca\u00b2\u207a]i (mM)"),
}

def run_other_experiment(panels=None, **overrides):
    p = {**DEFAULT_OTHER_PARAMS, **overrides}
    _apply_cell_params(p)
    _reset_stimuli()  # a previous basic/voltage-clamp/synaptic run must not stay active here

    stim.delay = p["stim_delay"]
    stim.dur = p["stim_dur"]
    stim.amp = p["stim_amp"]

    stim2.delay = p["stim2_delay"]
    stim2.dur = p["stim2_dur"]
    stim2.amp = p["stim2_amp"]

    vclamp.dur[0] = p["vc_dur0"]
    vclamp.dur[1] = p["vc_dur1"]
    vclamp.dur[2] = p["vc_dur2"]
    vclamp.amp[0] = p["vc_amp0"]
    vclamp.amp[1] = p["vc_amp1"]
    vclamp.amp[2] = p["vc_amp2"]

    n.tstop = p["tstop"]
    n.v_init = p["v_init"]
    n.run()

    _plot_panels(PLOT_PANELS_OTHER, panels)


In [ ]:
# @title Specifying default parameters

OTHER_EXPERIMENTS = {
    "iA: action potential (exp. 11)": dict(
        pna_leak=6.9e-8, pk_leak=4.14e-7,
        gnabar_HH=0.0517, gkbar_HH=0.0069, gkbar_iA=0.00345,
        nai=31, nao=145, ki=135, ko=3.1,
        v_init=-65, tstop=60,
    ),
    "iA: voltage clamp (exp. 11)": dict(
        gkbar_iA=0.00345,
        nai=30, nao=145, ki=135, ko=3.1,
        celsius=23.5,
        vc_dur0=10, vc_dur1=100, vc_amp0=-100, vc_amp1=0,
        v_init=-100, tstop=100,
    ),
    "iL & iC (exp. 12)": dict(
        pna_leak=4.31e-8, pk_leak=4.14e-7,
        gnabar_HH=0.0517, gkbar_HH=0.00345, gkbar_iC=0.00345, pcabar_iL=0.000276,
        nai=31, nao=145, ki=135, ko=3.1, cai0_ca_ion=5e-5, cao0_ca_ion=2,
        stim_delay=3, stim_dur=3, stim_amp=1,
        v_init=-55, tstop=30,
    ),
    "iAHP (exp. 13)": dict(
        pna_leak=2e-8, pk_leak=4.14e-7, pcabar_iL=0.000276,
        gnabar_HH=0.0517, gkbar_HH=0.0069, gkbar_iAHP=0.000207, gkbar_iC=0.00345,
        nai=31, nao=145, ki=135, ko=3.1, cai0_ca_ion=5e-5, cao0_ca_ion=2,
        stim_delay=50, stim_dur=300, stim_amp=0.6,
        v_init=-70, tstop=600,
    ),
    "iT (exp. 14)": dict(
        pna_leak=2.7414e-8, pk_leak=5.069e-7,
        pcabar_iL=0.00027586, pcabar_iT=0.00010345, gkbar_iC=0.0068966, gkbar_iA=0.0034483,
        gnabar_HH=0.051724, gkbar_HH=0.0068966,
        nai=31, nao=145, ki=135, ko=3.1, cai0_ca_ion=5e-5, cao0_ca_ion=2,
        stim2_delay=0, stim2_dur=9999, stim2_amp=-0.27,
        stim_delay=25, stim_dur=200, stim_amp=0.12,
        v_init=-85, tstop=300,
    ),
    "iM: current clamp (exp. 15)": dict(
        pna_leak=1.7241e-8, pk_leak=4.1379e-7, gkbar_iM=0.00031035,
        gnabar_HH=0.051724, gkbar_HH=0.0068966,
        nai=31, nao=145, ki=135, ko=3.1,
        stim_delay=50, stim_dur=150, stim_amp=0.7,
        v_init=-75, tstop=300,
    ),
    "iM: voltage clamp (exp. 15)": dict(
        pna_leak=3.4483e-8, pk_leak=3.4483e-7, gkbar_iM=0.00086207,
        nai=30, nao=145, ki=135, ko=3.1,
        vc_dur0=50, vc_dur1=250, vc_dur2=100, vc_amp0=-70, vc_amp1=-30, vc_amp2=-70,
        v_init=-70, tstop=400,
    ),
}

OTHER_PARAM_UI = {
    "celsius": dict(min=0, max=40, step=0.5, description="celsius", unit="\u00b0C"),
    "pna_leak": dict(min=0, max=8, step=0.01, readout_format=".2f", description="pNa (leak)", unit="\u00d710\u207b\u2077 cm/s", scale=1e-7),
    "pk_leak": dict(min=0, max=8, step=0.01, readout_format=".2f", description="pK (leak)", unit="\u00d710\u207b\u2076 cm/s", scale=1e-6),
    "gnabar_HH": dict(min=0, max=0.15, step=0.001, readout_format=".3f", description="gNa (HH)", unit="S/cm\u00b2"),
    "gkbar_HH": dict(min=0, max=0.02, step=0.0005, readout_format=".4f", description="gK (HH)", unit="S/cm\u00b2"),
    "gkbar_iA": dict(min=0, max=0.02, step=0.0005, readout_format=".4f", description="gK (iA)", unit="S/cm\u00b2"),
    "pcabar_iL": dict(min=0, max=0.001, step=0.00001, readout_format=".5f", description="pCa (iL)", unit="cm/s"),
    "pcabar_iT": dict(min=0, max=0.001, step=0.00001, readout_format=".5f", description="pCa (iT)", unit="cm/s"),
    "gkbar_iC": dict(min=0, max=0.02, step=0.0005, readout_format=".4f", description="gK (iC)", unit="S/cm\u00b2"),
    "gkbar_iAHP": dict(min=0, max=0.001, step=0.00001, readout_format=".5f", description="gK (iAHP)", unit="S/cm\u00b2"),
    "gkbar_iM": dict(min=0, max=0.002, step=0.00001, readout_format=".5f", description="gK (iM)", unit="S/cm\u00b2"),
    "nai": dict(min=0, max=60, step=1, description="nai", unit="mM"),
    "nao": dict(min=0, max=200, step=1, description="nao", unit="mM"),
    "ki": dict(min=0, max=200, step=1, description="ki", unit="mM"),
    "ko": dict(min=0, max=10, step=0.1, description="ko", unit="mM"),
    "stim_delay": dict(min=0, max=50, step=1, description="stim delay", unit="ms"),
    "stim_dur": dict(min=0, max=300, step=1, description="stim dur", unit="ms"),
    "stim_amp": dict(min=-1, max=2, step=0.01, description="stim amp", unit="nA"),
    "stim2_delay": dict(min=0, max=50, step=1, description="stim2 delay", unit="ms"),
    "stim2_dur": dict(min=0, max=9999, step=1, description="stim2 dur", unit="ms"),
    "stim2_amp": dict(min=-1, max=1, step=0.01, description="stim2 amp", unit="nA"),
    "vc_dur0": dict(min=0, max=250, step=1, description="dur0 (hold)", unit="ms"),
    "vc_dur1": dict(min=0, max=250, step=1, description="dur1 (step)", unit="ms"),
    "vc_dur2": dict(min=0, max=250, step=1, description="dur2 (return)", unit="ms"),
    "vc_amp0": dict(min=-100, max=50, step=1, description="amp0 (hold)", unit="mV"),
    "vc_amp1": dict(min=-100, max=50, step=1, description="amp1 (step)", unit="mV"),
    "vc_amp2": dict(min=-100, max=50, step=1, description="amp2 (return)", unit="mV"),
    "v_init": dict(min=-100, max=-30, step=1, description="v_init", unit="mV"),
    "tstop": dict(min=5, max=600, step=5, description="tstop", unit="ms"),
}

def build_otherExp_dashboard():
    def _preset_for(exp_name):
        return {**DEFAULT_OTHER_PARAMS, **OTHER_EXPERIMENTS[exp_name]}

    exp_selector = Dropdown(options=list(OTHER_EXPERIMENTS.keys()), description="Experiment:",
                             style={"description_width": "initial"}, layout=Layout(width="320px"))
    current_defaults = _preset_for(exp_selector.value)

    panel_checks, panels_row = _build_panel_checks(PLOT_PANELS_OTHER)
    sliders, changed_flags, rows = _build_param_rows(OTHER_PARAM_UI, lambda name: current_defaults[name])

    run_button = Button(description="Run", button_style="success", icon="play")
    reset_all_button = Button(description="Reset all", icon="undo")
    output = Output()

    def _on_run(_btn):
        with output:
            output.clear_output(wait=True)
            run_other_experiment(panels=_selected_panels(panel_checks), **_raw_values(OTHER_PARAM_UI, sliders))

    def _on_reset_all(_btn):
        for name, slider in sliders.items():
            slider.value = _display_value(OTHER_PARAM_UI, name, current_defaults[name])

    def _on_experiment_change(change):
        nonlocal current_defaults
        current_defaults = _preset_for(change["new"])
        _on_reset_all(None)
        _on_run(None)

    run_button.on_click(_on_run)
    reset_all_button.on_click(_on_reset_all)
    exp_selector.observe(_on_experiment_change, names="value")

    dashboard = VBox([exp_selector, HBox([VBox([panels_row, output]),
                                          VBox([HBox([run_button, reset_all_button])] + rows)])])
    _on_run(None)  # show a default run immediately, without waiting for a click
    return dashboard



Pick an experiment from the dropdown, then adjust sliders and click **Run**. "iA: action
potential" starts with `stim amp` at 0 &mdash; raise it to trigger a spike and see how the
A-current delays the first spike. "iT" drives `stim2` as a constant hyperpolarizing base
current (mimicking the book's rebound-burst protocol) plus a depolarizing pulse on `stim`;
the voltage-clamp presets ("iA"/"iM") instead step `vclamp` through `amp0`/`amp1`/`amp2`.



In [ ]:
display(build_otherExp_dashboard())


# 5. Synaptic experiments (exp. 16-17)

Four synapse point processes sit on the same `soma`, each an alpha-function-shaped
conductance triggered at a fixed `onset` time (no presynaptic spike/pointer needed for
this single-cell demo): `AmpaSynapse`/`NmdaSynapse` produce an e.p.s.p., `GABAaSynapse`/
`GABAbSynapse` an i.p.s.p. `gmax_EPSP`/`onset_EPSP` drive both excitatory synapses at once
(scaled per-synapse by `.w`); `gmax_IPSP`/`onset_IPSP` do the same for the inhibitory pair.
`NmdaSynapse` also reads `mgo` (Mg block) and `GABAaSynapse` reads `cli`/`clo` (via `ecl`),
so this section additionally sets those ion concentrations. Sourced directly from `e16a.par`,
`e16b.par`, `e16c.par`, `e17a.par` and `e17b.par`.


In [ ]:
# synapses, specific to experiments 16-17
ampasyn = n.AmpaSynapse(soma(0.5))
nmdasyn = n.NmdaSynapse(soma(0.5))
gabaAsyn = n.GABAaSynapse(soma(0.5))
gabaBsyn = n.GABAbSynapse(soma(0.5))

ampasyn_i = n.Vector().record(ampasyn._ref_i)
nmdasyn_i = n.Vector().record(nmdasyn._ref_i)
gabaAsyn_i = n.Vector().record(gabaAsyn._ref_i)
gabaBsyn_i = n.Vector().record(gabaBsyn._ref_i)

# synaptic-specific defaults, layered on top of the common cell/ion defaults; also carries
# the current-clamp and voltage-clamp keys (defaulting to "off") since e17b uses a base
# current (stim) and e16c uses a voltage-clamp protocol (vclamp) alongside the synapses
DEFAULT_SYN_PARAMS = {**DEFAULT_PARAMS,
    "gmax_EPSP": 0.0, "onset_EPSP": 20.0, "ampasyn_w": 1.0, "nmdasyn_w": 1.0,
    "gmax_IPSP": 0.0, "onset_IPSP": 25.0, "gabaAsyn_w": 1.0, "gabaBsyn_w": 0.05,
    "stim_delay": 0.0, "stim_dur": 0.0, "stim_amp": 0.0,
    "vc_dur0": 0.0, "vc_dur1": 0.0, "vc_dur2": 0.0,
    "vc_amp0": 0.0, "vc_amp1": 0.0, "vc_amp2": 0.0,
    "v_init": -55.0, "tstop": 500.0,
}

PLOT_PANELS_SYN = {
    "v": dict(label="voltage", vector=v_vec, ylabel="membrane potential (mV)", ylim=(-100, 50)),
    "isyn": dict(label="synaptic current",
                 compute=lambda: ampasyn_i.c().add(nmdasyn_i).add(gabaAsyn_i).add(gabaBsyn_i),
                 ylabel="ampasyn.i+gabaAsyn.i+gabaBsyn.i+nmdasyn.i (nA)"),
}

def run_syn_experiment(panels=None, **overrides):
    p = {**DEFAULT_SYN_PARAMS, **overrides}
    _apply_cell_params(p)
    _reset_stimuli()  # a previous current-clamp/voltage-clamp run must not stay active here

    stim.delay = p["stim_delay"]
    stim.dur = p["stim_dur"]
    stim.amp = p["stim_amp"]

    vclamp.dur[0] = p["vc_dur0"]
    vclamp.dur[1] = p["vc_dur1"]
    vclamp.dur[2] = p["vc_dur2"]
    vclamp.amp[0] = p["vc_amp0"]
    vclamp.amp[1] = p["vc_amp1"]
    vclamp.amp[2] = p["vc_amp2"]

    ampasyn.gmaxEPSP = nmdasyn.gmaxEPSP = p["gmax_EPSP"]
    ampasyn.onset = nmdasyn.onset = p["onset_EPSP"]
    ampasyn.w = p["ampasyn_w"]
    nmdasyn.w = p["nmdasyn_w"]

    gabaAsyn.gmaxIPSP = gabaBsyn.gmaxIPSP = p["gmax_IPSP"]
    gabaAsyn.onset = gabaBsyn.onset = p["onset_IPSP"]
    gabaAsyn.w = p["gabaAsyn_w"]
    gabaBsyn.w = p["gabaBsyn_w"]

    n.tstop = p["tstop"]
    n.v_init = p["v_init"]
    n.run()

    _plot_panels(PLOT_PANELS_SYN, panels)


In [ ]:
# @title Specifying default parameters

SYN_EXPERIMENTS = {
    "1: e.p.s.p. (exp. 16)": dict(
        pna_leak=3.4138e-8, pk_leak=3.4483e-7,
        nai=31, nao=145, ki=135, ko=3.1, cli=7, clo=120, mgo=1.2,
        gmax_EPSP=0.02, onset_EPSP=20, ampasyn_w=1, nmdasyn_w=0.5,
        gmax_IPSP=0.1, onset_IPSP=25, gabaAsyn_w=1, gabaBsyn_w=0.05,
        v_init=-55, tstop=500,
    ),
    "2: nmda current (exp. 16)": dict(
        pna_leak=4.069e-8, pk_leak=4.1379e-7,
        nai=31, nao=145, ki=135, ko=3.1, cli=7, clo=120, mgo=1.2,
        gmax_EPSP=0.02, onset_EPSP=10, ampasyn_w=0, nmdasyn_w=1,
        gmax_IPSP=0, onset_IPSP=25, gabaAsyn_w=1, gabaBsyn_w=0.05,
        v_init=-55, tstop=120,
    ),
    "3: nmda voltage clamp (exp. 16)": dict(
        nai=30, nao=145, ki=135, ko=3.1, cli=8, clo=140, mgo=1,
        vc_dur0=2, vc_dur1=28, vc_dur2=0, vc_amp0=-100, vc_amp1=-20, vc_amp2=0,
        gmax_EPSP=0.1, onset_EPSP=1, ampasyn_w=0, nmdasyn_w=1,
        gmax_IPSP=0, onset_IPSP=25, gabaAsyn_w=1, gabaBsyn_w=0.05,
        v_init=-55, tstop=30,
    ),
    "4: i.p.s.p. (exp. 17)": dict(
        pna_leak=3.3793e-8, pk_leak=3.4483e-7,
        nai=31, nao=145, ki=135, ko=3.1, cli=7, clo=120, mgo=1.2,
        gmax_EPSP=0, onset_EPSP=20, ampasyn_w=1, nmdasyn_w=0.5,
        gmax_IPSP=0.1, onset_IPSP=25, gabaAsyn_w=1, gabaBsyn_w=0.05,
        v_init=-55, tstop=500,
    ),
    "5: i.p.s.p. + e.p.s.p. (exp. 17)": dict(
        pna_leak=6.0345e-8, pk_leak=5.5172e-7,
        gnabar_HH=0.051724, gkbar_HH=0.0068966, gkbar_iA=0.0034483,
        nai=31, nao=145, ki=135, ko=3.1, cli=7, clo=120, mgo=1,
        stim_delay=0, stim_dur=9999, stim_amp=-0.69,
        gmax_EPSP=0.15, onset_EPSP=20, ampasyn_w=1, nmdasyn_w=1,
        gmax_IPSP=0, onset_IPSP=21, gabaAsyn_w=1, gabaBsyn_w=0,
        v_init=-85, tstop=100,
    ),
}

SYN_PARAM_UI = {
    "pna_leak": dict(min=0, max=8, step=0.01, readout_format=".2f", description="pNa (leak)", unit="\u00d710\u207b\u2077 cm/s", scale=1e-7),
    "pk_leak": dict(min=0, max=8, step=0.01, readout_format=".2f", description="pK (leak)", unit="\u00d710\u207b\u2076 cm/s", scale=1e-6),
    "gnabar_HH": dict(min=0, max=0.15, step=0.001, readout_format=".3f", description="gNa (HH)", unit="S/cm\u00b2"),
    "gkbar_HH": dict(min=0, max=0.02, step=0.0005, readout_format=".4f", description="gK (HH)", unit="S/cm\u00b2"),
    "gkbar_iA": dict(min=0, max=0.02, step=0.0005, readout_format=".4f", description="gK (iA)", unit="S/cm\u00b2"),
    "nai": dict(min=0, max=60, step=1, description="nai", unit="mM"),
    "nao": dict(min=0, max=200, step=1, description="nao", unit="mM"),
    "ki": dict(min=0, max=200, step=1, description="ki", unit="mM"),
    "ko": dict(min=0, max=10, step=0.1, description="ko", unit="mM"),
    "cli": dict(min=0, max=20, step=0.5, description="cli", unit="mM"),
    "clo": dict(min=0, max=200, step=1, description="clo", unit="mM"),
    "mgo": dict(min=0, max=5, step=0.1, description="mgo", unit="mM"),
    "gmax_EPSP": dict(min=0, max=0.3, step=0.005, readout_format=".3f", description="gmax EPSP", unit="nS"),
    "onset_EPSP": dict(min=0, max=100, step=1, description="onset EPSP", unit="ms"),
    "ampasyn_w": dict(min=0, max=2, step=0.05, description="ampasyn.w", unit=""),
    "nmdasyn_w": dict(min=0, max=2, step=0.05, description="nmdasyn.w", unit=""),
    "gmax_IPSP": dict(min=0, max=0.3, step=0.005, readout_format=".3f", description="gmax IPSP", unit="nS"),
    "onset_IPSP": dict(min=0, max=100, step=1, description="onset IPSP", unit="ms"),
    "gabaAsyn_w": dict(min=0, max=2, step=0.05, description="gabaAsyn.w", unit=""),
    "gabaBsyn_w": dict(min=0, max=2, step=0.05, description="gabaBsyn.w", unit=""),
    "stim_delay": dict(min=0, max=50, step=1, description="stim delay", unit="ms"),
    "stim_dur": dict(min=0, max=9999, step=1, description="stim dur", unit="ms"),
    "stim_amp": dict(min=-2, max=2, step=0.01, description="stim amp", unit="nA"),
    "vc_dur0": dict(min=0, max=50, step=1, description="dur0 (hold)", unit="ms"),
    "vc_dur1": dict(min=0, max=50, step=1, description="dur1 (step)", unit="ms"),
    "vc_dur2": dict(min=0, max=50, step=1, description="dur2 (return)", unit="ms"),
    "vc_amp0": dict(min=-100, max=50, step=1, description="amp0 (hold)", unit="mV"),
    "vc_amp1": dict(min=-100, max=50, step=1, description="amp1 (step)", unit="mV"),
    "vc_amp2": dict(min=-100, max=50, step=1, description="amp2 (return)", unit="mV"),
    "v_init": dict(min=-100, max=-30, step=1, description="v_init", unit="mV"),
    "tstop": dict(min=5, max=500, step=5, description="tstop", unit="ms"),
}

def build_synExp_dashboard():
    def _preset_for(exp_name):
        return {**DEFAULT_SYN_PARAMS, **SYN_EXPERIMENTS[exp_name]}

    exp_selector = Dropdown(options=list(SYN_EXPERIMENTS.keys()), description="Experiment:",
                             style={"description_width": "initial"}, layout=Layout(width="320px"))
    current_defaults = _preset_for(exp_selector.value)

    panel_checks, panels_row = _build_panel_checks(PLOT_PANELS_SYN)
    sliders, changed_flags, rows = _build_param_rows(SYN_PARAM_UI, lambda name: current_defaults[name])

    run_button = Button(description="Run", button_style="success", icon="play")
    reset_all_button = Button(description="Reset all", icon="undo")
    output = Output()

    def _on_run(_btn):
        with output:
            output.clear_output(wait=True)
            run_syn_experiment(panels=_selected_panels(panel_checks), **_raw_values(SYN_PARAM_UI, sliders))

    def _on_reset_all(_btn):
        for name, slider in sliders.items():
            slider.value = _display_value(SYN_PARAM_UI, name, current_defaults[name])

    def _on_experiment_change(change):
        nonlocal current_defaults
        current_defaults = _preset_for(change["new"])
        _on_reset_all(None)
        _on_run(None)

    run_button.on_click(_on_run)
    reset_all_button.on_click(_on_reset_all)
    exp_selector.observe(_on_experiment_change, names="value")

    dashboard = VBox([exp_selector, HBox([VBox([panels_row, output]),
                                          VBox([HBox([run_button, reset_all_button])] + rows)])])
    _on_run(None)  # show a default run immediately, without waiting for a click
    return dashboard


Pick an experiment from the dropdown, then adjust sliders and click **Run**. "e.p.s.p." and
"i.p.s.p." each fire both a real synapse (e.g. `ampasyn`) and its counterpart at zero gain, so
you can raise `gmax_IPSP`/`gmax_EPSP` to see them interact; "nmda voltage clamp" drives `vclamp`
instead of `stim` to isolate the NMDA current's voltage/Mg-dependence.


In [ ]:
display(build_synExp_dashboard())
